# 02 — Training
Train BiLSTM baseline, then SPOTER. Save checkpoints to Drive.

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/isl_project'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

!pip install mediapipe opencv-python-headless scipy tqdm pyyaml scikit-learn seaborn -q

In [ ]:
# Quick dataset sanity check
from dataset import ISLDataset
from torch.utils.data import DataLoader

SPLITS_DIR = f'{PROJECT_ROOT}/data/splits'
NPY_DIR = f'{PROJECT_ROOT}/data/keypoints'

ds = ISLDataset(
    npy_dir=NPY_DIR,
    split_file=f'{SPLITS_DIR}/train.txt',
    mean_path=f'{SPLITS_DIR}/mean.npy',
    std_path=f'{SPLITS_DIR}/std.npy',
    augment=True
)
print(f'Train samples: {len(ds)}')
x, y = ds[0]
print(f'Sample shape: {x.shape}, label: {y}')  # expect (64, 543), int

In [ ]:
# Train BiLSTM (~2.5 hours on T4)
from train import train
train(f'{PROJECT_ROOT}/configs/bilstm.yaml')

In [ ]:
# Train SPOTER (~10 hours on T4)
from train import train
train(f'{PROJECT_ROOT}/configs/spoter.yaml')

In [ ]:
# Monitor training (run periodically)
import json, matplotlib.pyplot as plt

def plot_log(log_path, title):
    with open(log_path) as f:
        log = json.load(f)
    epochs = [e['epoch'] for e in log]
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, [e['train_loss'] for e in log], label='train')
    plt.plot(epochs, [e['val_loss'] for e in log], label='val')
    plt.legend(); plt.title(f'{title} Loss')
    plt.subplot(1, 2, 2)
    plt.plot(epochs, [e['val_acc'] for e in log], label='val top-1')
    plt.plot(epochs, [e['val_top5'] for e in log], label='val top-5')
    plt.legend(); plt.title(f'{title} Accuracy')
    plt.tight_layout(); plt.show()

plot_log(f'{PROJECT_ROOT}/checkpoints/bilstm/log.json', 'BiLSTM')
# plot_log(f'{PROJECT_ROOT}/checkpoints/spoter/log.json', 'SPOTER')